In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import config


SPREAD_WINDOW = 20
SPREAD_MIN_OBS = 12
ADV_WINDOW = 126

YAHOO_INPUT_DIR = config.INPUT_DATA_DIR / "yahoo"


def split_adjust_ohlc(data):
    """
    Split-adjust High, Low and Close without dividend-adjusting them.

    Corwin-Schultz compares prices across consecutive trading days.
    A stock/ETF split must therefore not appear as a genuine price jump.
    """

    high = pd.to_numeric(data["High"], errors="coerce")
    low = pd.to_numeric(data["Low"], errors="coerce")
    close = pd.to_numeric(data["Close"], errors="coerce")

    if "Stock Splits" not in data.columns:
        return high, low, close

    splits = pd.to_numeric(
        data["Stock Splits"],
        errors="coerce",
    ).fillna(0.0)

    split_ratios = splits.where(
        splits != 0,
        1.0,
    )

    # Product of all splits strictly after each observation.
    future_split_factor = (
        split_ratios.iloc[::-1]
        .cumprod()
        .iloc[::-1]
        .shift(-1, fill_value=1.0)
    )

    return (
        high / future_split_factor,
        low / future_split_factor,
        close / future_split_factor,
    )


def calculate_corwin_schultz_spread(data):
    """
    Calculate the Corwin-Schultz proportional full bid-ask spread.

    The raw estimator uses two adjacent trading days.
    Negative estimates are set to zero.

    The final Spread column is the trailing 20-trading-day average
    of those estimates, requiring at least 12 valid estimates.
    """

    high, low, close = split_adjust_ohlc(data)

    volume = pd.to_numeric(
        data["Volume"],
        errors="coerce",
    )

    valid = (
        (high > low)
        & (low > 0)
        & (close > 0)
        & (volume > 0)
    )

    high = high.where(valid)
    low = low.where(valid)
    close = close.where(close > 0)

    # Values from the previous trading day.
    prev_high = high.shift(1)
    prev_low = low.shift(1)
    prev_close = close.shift(1)

    adjusted_high = high.copy()
    adjusted_low = low.copy()

    # -------------------------------------------------------------------------
    # Remove upward overnight gaps
    # -------------------------------------------------------------------------

    gap_up = (
        (prev_close > 0)
        & (prev_close < low)
    )

    adjusted_high = adjusted_high.where(
        ~gap_up,
        high - (low - prev_close),
    )

    adjusted_low = adjusted_low.where(
        ~gap_up,
        prev_close,
    )

    # -------------------------------------------------------------------------
    # Remove downward overnight gaps
    # -------------------------------------------------------------------------

    gap_down = (
        (prev_close > 0)
        & (prev_close > high)
    )

    adjusted_high = adjusted_high.where(
        ~gap_down,
        prev_close,
    )

    adjusted_low = adjusted_low.where(
        ~gap_down,
        low + (prev_close - high),
    )

    valid_pair = (
        (adjusted_high > adjusted_low)
        & (adjusted_low > 0)
        & (prev_high > prev_low)
        & (prev_low > 0)
    )

    adjusted_high = adjusted_high.where(
        valid_pair
    )

    adjusted_low = adjusted_low.where(
        valid_pair
    )

    # -------------------------------------------------------------------------
    # Corwin-Schultz beta
    # -------------------------------------------------------------------------

    beta = (
        np.log(
            adjusted_high / adjusted_low
        ).pow(2)
        +
        np.log(
            prev_high / prev_low
        ).pow(2)
    )

    # -------------------------------------------------------------------------
    # Corwin-Schultz gamma
    # -------------------------------------------------------------------------

    two_day_high = pd.concat(
        [
            adjusted_high,
            prev_high,
        ],
        axis=1,
    ).max(
        axis=1,
        skipna=False,
    )

    two_day_low = pd.concat(
        [
            adjusted_low,
            prev_low,
        ],
        axis=1,
    ).min(
        axis=1,
        skipna=False,
    )

    gamma = np.log(
        two_day_high / two_day_low
    ).pow(2)

    # -------------------------------------------------------------------------
    # Alpha and spread
    # -------------------------------------------------------------------------

    constant = (
        3
        - 2 * np.sqrt(2)
    )

    alpha = (
        (
            np.sqrt(2 * beta)
            - np.sqrt(beta)
        )
        / constant
        -
        np.sqrt(
            gamma / constant
        )
    )

    raw_spread = (
        2
        * (np.exp(alpha) - 1)
        / (1 + np.exp(alpha))
    ).clip(
        lower=0
    )

    # Smooth the noisy two-day estimates.
    spread = raw_spread.rolling(
        window=SPREAD_WINDOW,
        min_periods=SPREAD_MIN_OBS,
    ).mean()

    return spread


def calculate_adv_dollars(data):
    """
    Calculate six-month Average Daily Dollar Volume.

    Dollar Volume_t:
        Close_t * Volume_t

    ADV_t:
        average Dollar Volume over the previous 126 trading days,
        including date t.
    """

    close = pd.to_numeric(
        data["Close"],
        errors="coerce",
    )

    volume = pd.to_numeric(
        data["Volume"],
        errors="coerce",
    )

    dollar_volume = (
        close * volume
    ).where(
        (close > 0)
        & (volume > 0)
    )

    adv_dollars = dollar_volume.rolling(
        window=ADV_WINDOW,
        min_periods=ADV_WINDOW,
    ).mean()

    return adv_dollars


def add_liquidity_columns(
    file_path,
    asset_class,
):
    """
    Add Spread and ADV_Dollars to one cleaned Yahoo OHLCV file.

    The existing CSV is overwritten with the two additional columns.
    """

    data = pd.read_csv(
        file_path
    )

    data["Date"] = pd.to_datetime(
        data["Date"]
    )

    data = (
        data
        .sort_values("Date")
        .drop_duplicates(
            "Date",
            keep="last",
        )
        .reset_index(
            drop=True
        )
    )

    required_columns = {
        "High",
        "Low",
        "Close",
        "Volume",
    }

    missing_columns = (
        required_columns
        - set(data.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{file_path.stem}: missing columns "
            f"{sorted(missing_columns)}"
        )

    # We are deliberately not applying the equity/ETF
    # microstructure model to Bitcoin.
    if (
        str(asset_class)
        .strip()
        .lower()
        == "crypto"
    ):

        data["Spread"] = np.nan
        data["ADV_Dollars"] = np.nan

    else:

        data["Spread"] = (
            calculate_corwin_schultz_spread(
                data
            )
        )

        data["ADV_Dollars"] = (
            calculate_adv_dollars(
                data
            )
        )

    data.to_csv(
        file_path,
        index=False,
    )

    print(
        f"{file_path.stem}: "
        f"{data['Spread'].notna().sum():,} spread values, "
        f"{data['ADV_Dollars'].notna().sum():,} ADV values"
    )


def main():

    universe = pd.read_csv(
        config.UNIVERSE_FILE
    )

    yahoo_assets = universe[
        universe["Source"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("yahoo finance")
    ][
        [
            "Ticker",
            "Asset Class",
        ]
    ].drop_duplicates(
        "Ticker"
    )

    for _, asset in yahoo_assets.iterrows():

        ticker = asset["Ticker"]

        file_path = (
            Path(YAHOO_INPUT_DIR)
            / f"{ticker}.csv"
        )

        if not file_path.exists():

            raise FileNotFoundError(
                f"Missing Yahoo file: "
                f"{file_path}"
            )

        add_liquidity_columns(
            file_path=file_path,
            asset_class=asset["Asset Class"],
        )

    print(
        "\nSpread and ADV_Dollars added "
        "to all Yahoo OHLCV files."
    )


if __name__ == "__main__":
    main()

SPY: 8,458 spread values, 8,345 ADV values
VT: 4,576 spread values, 4,463 ADV values
QQQ: 6,916 spread values, 6,803 ADV values
VTV: 5,685 spread values, 5,572 ADV values
VUG: 5,683 spread values, 5,534 ADV values
IWM: 6,608 spread values, 6,495 ADV values
DVY: 5,741 spread values, 5,628 ADV values
VEA: 4,808 spread values, 4,695 ADV values
VGK: 5,406 spread values, 5,293 ADV values
FEZ: 6,004 spread values, 5,839 ADV values
EWJ: 7,663 spread values, 7,553 ADV values
FXI: 5,511 spread values, 5,398 ADV values
AFK: 4,550 spread values, 4,194 ADV values
VWO: 5,406 spread values, 5,293 ADV values
IPO: 3,226 spread values, 2,648 ADV values
IXP: 5,863 spread values, 5,503 ADV values
SMH: 6,603 spread values, 6,490 ADV values
IXG: 5,859 spread values, 5,499 ADV values
IXN: 5,916 spread values, 5,572 ADV values
XBI: 5,177 spread values, 5,064 ADV values
IXJ: 6,189 spread values, 5,869 ADV values
PBW: 5,411 spread values, 5,298 ADV values
VNQ: 5,518 spread values, 5,405 ADV values
RWX: 4,957 s